# rPPG Clinical Model - Training on Google Colab
Training notebook for remote photoplethysmography model on MCD-rPPG dataset with GPU.

In [ ]:
!git clone https://github.com/kpanuragh/facescan.git
%cd facescan

In [ ]:
!pip install -q torch torchvision opencv-python mediapipe scikit-learn pandas scipy datasets huggingface-hub fastapi uvicorn pydantic pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
!python -m pytest tests/test_preprocessing.py -v

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import sys
sys.path.insert(0, '/content/facescan')
from src.preprocessing.video_loader import VideoLoader
loader = VideoLoader()
dataset = loader.load_dataset()
print(f'Dataset: {len(dataset)} samples')

In [ ]:
from src.preprocessing.dataset_builder import create_splits
train_idx, val_idx, test_idx = create_splits(dataset)
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}')

In [ ]:
from src.models.architecture import rPPGModel
from src.config import FEATURE_DIM, HIDDEN_DIM, DEVICE
model = rPPGModel(feature_dim=FEATURE_DIM, hidden_dim=HIDDEN_DIM).to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'Model: {params:,} parameters')

In [ ]:
from src.training.trainer import Trainer
from src.config import BATCH_SIZE, LEARNING_RATE, EPOCHS, CHECKPOINTS_DIR
import torch.optim as optim
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)
trainer = Trainer(model, DEVICE, checkpoint_dir=CHECKPOINTS_DIR)
print('Trainer ready - run trainer.train() to start')